# SRJ Score Plotting

Visualise and compare tagger scores produced by Step 4 (`test_make_scores_SRJ.py`).  
Typical plots:
- Score distributions (quark vs gluon)
- ROC curve: quark efficiency vs gluon rejection (with ratio panel)
- Rejection at fixed efficiency vs jet pT
- Score correlation between models
- Confusion matrix

**Before running:** fill in the path variables in the *Configuration* cell below.

In [ ]:
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from sklearn.metrics import roc_curve, auc

from tools.GNN_model_weight.utils_newdata import JSD
from plotting.utils_plots_matplotlib import hist_with_errors

## Configuration

Fill in the paths and model names below. Everything else runs automatically.

In [ ]:
# ── Input files ────────────────────────────────────────────────────────────────
# ROOT file(s) produced by test_make_scores_SRJ.py.
# Supports glob patterns (e.g. "/out/scores/*.root").
SCORE_FILES = [
    "/path/to/your/scores/Final_Scores_test_kTNone.root",
]

# Cross-section metadata for per-DSID physics weights.
# The repo ships metadata.json with xsec + genFiltEff for DSIDs 364700-364712.
# sumWeights is pre-filled for 364700-364702; for other DSIDs set to None and
# the notebook will skip xsec weighting for those DSIDs (mcEventWeight only).
# Set METADATA_JSON = None to disable all physics weighting.
import pathlib
_cwd = pathlib.Path.cwd()
# If the kernel was launched from repo root, metadata.json is here;
# if launched from notebooks/, it is one level up.
if (_cwd / "metadata.json").exists():
    _REPO_ROOT = _cwd
elif (_cwd.parent / "metadata.json").exists():
    _REPO_ROOT = _cwd.parent
else:
    _REPO_ROOT = _cwd.parent  # fallback: assume notebooks/ subdir
METADATA_JSON = str(_REPO_ROOT / "metadata.json")

# Tree name in the ROOT files
TREE_NAME = "FlatSubstructureJetTree"

# ── Models ─────────────────────────────────────────────────────────────────────
# Map display name → score branch name in the ROOT file.
# Branch names follow the pattern: fjet_{tag}_score  (tag set in config_make_scores_SRJ.yaml)
MODELS = {
    "QLundNet": "fjet_LundNet_SRJ_score",   # adjust tags to match your config
    "LundNet":  "fjet_LundNet_SRJ_score",   # example: same branch, rename as needed
    # "Combiner": "fjet_Combined_score",
}

# Reference model for ratio panels
REFERENCE_MODEL = "LundNet"

# ── Kinematic selection ────────────────────────────────────────────────────────
PT_MIN, PT_MAX   = 20, 160     # GeV
ETA_MIN, ETA_MAX = 3.2, 4.5

# ── Colours ────────────────────────────────────────────────────────────────────
MODEL_COLORS = {
    "QLundNet": "tab:blue",
    "LundNet":  "tab:purple",
    "Combiner": "tab:orange",
}

## Load data

In [ ]:
import glob, json

# Expand glob patterns
score_paths = []
for pat in SCORE_FILES:
    score_paths.extend(sorted(glob.glob(pat)))
if not score_paths:
    raise FileNotFoundError(f"No files matched: {SCORE_FILES}")
print(f"Loading {len(score_paths)} ROOT file(s)...")

paths_with_tree = [f"{p}:{TREE_NAME}" for p in score_paths]
data = uproot.concatenate(paths_with_tree)

print(f"Loaded {len(data)} jets")
print(f"  signal (quark): {int(np.sum(data['labels'] == 1))}")
print(f"  background (gluon): {int(np.sum(data['labels'] == 0))}")
print("Available branches:", data.fields)

In [ ]:
# ── Event weights ──────────────────────────────────────────────────────────────
# Try to build per-DSID physics weights from the metadata JSON.
# Falls back to mcEventWeight-only if JSON is unavailable or DSID is missing.

dsid_factor = {}   # DSID → xsec * genFiltEff * kfactor / sumWeights

if METADATA_JSON is not None:
    try:
        with open(METADATA_JSON) as f:
            meta = json.load(f)
        data_to_iterate = meta.get("by_dsid", meta)
        for dsid_str, val in data_to_iterate.items():
            try:
                dsid_int = int(dsid_str)
            except ValueError:
                continue
            v = val[0] if isinstance(val, list) else val
            xsec   = v.get("cross_section_pb", v.get("xsec", v.get("XSection_pb")))
            gfe    = v.get("genFiltEff", v.get("genfilteff", v.get("genFilterEff")))
            kf     = v.get("kfactor", v.get("kFactor", 1.0))
            sumw   = v.get("sum_of_weight", v.get("sumWeights"))
            if xsec is not None and gfe is not None and sumw:
                dsid_factor[dsid_int] = xsec * gfe * kf / sumw
        print(f"Loaded DSID weights for {len(dsid_factor)} DSIDs from metadata JSON.")
    except Exception as e:
        print(f"Warning: could not load metadata JSON ({e}). Using mcEventWeight only.")

dsids = np.array(data["EventInfo_mcChannelNumber"])
mc_w  = np.array(data["EventInfo_mcEventWeight"])

xsec_factors = np.array([dsid_factor.get(int(d), 1.0) for d in dsids])
weights = mc_w * xsec_factors
print(f"Weight range: [{weights.min():.3g}, {weights.max():.3g}]")

In [ ]:
# ── Kinematic selection ────────────────────────────────────────────────────────
pt  = np.array(data["fjet_pt"])
eta = np.array(data["fjet_eta"])
mass = np.array(data["fjet_m"])
labels = np.array(data["labels"])

sel = (pt > PT_MIN) & (pt < PT_MAX) & (np.abs(eta) > ETA_MIN) & (np.abs(eta) < ETA_MAX)
print(f"Jets after kinematic cuts: {sel.sum()} / {len(sel)}")

# Build model score dict (all models, full dataset — cuts applied per plot)
models = {}
for name, branch in MODELS.items():
    if branch in data.fields:
        models[name] = {"scores": np.array(data[branch])}
    else:
        print(f"WARNING: branch '{branch}' not found for model '{name}'")

# Apply selection
pt_sel      = pt[sel]
eta_sel     = eta[sel]
mass_sel    = mass[sel]
labels_sel  = labels[sel]
weights_sel = weights[sel]
for name in models:
    models[name]["scores_sel"] = models[name]["scores"][sel]

## Score distributions

In [ ]:
default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4), dpi=120, squeeze=False)

for ax, (name, m) in zip(axes[0], models.items()):
    color = MODEL_COLORS.get(name, default_colors[0])
    scores = m["scores_sel"]
    hist_kw = dict(bins=50, range=(0, 1), density=True, histtype="step", linewidth=2)
    ax.hist(scores[labels_sel == 1], weights=weights_sel[labels_sel == 1],
            label="Quark (signal)",    linestyle="-",  color=color, **hist_kw)
    ax.hist(scores[labels_sel == 0], weights=weights_sel[labels_sel == 0],
            label="Gluon (background)", linestyle="--", color=color, **hist_kw)
    ax.set_title(name)
    ax.set_xlabel("Score")
    ax.set_ylabel("Normalised entries")
    ax.legend(frameon=False, fontsize=9)

plt.suptitle(f"Score distributions  |  {PT_MIN} < pT < {PT_MAX} GeV, {ETA_MIN} < |η| < {ETA_MAX}")
plt.tight_layout()
plt.show()

## ROC curves — quark efficiency vs gluon rejection

In [ ]:
# ── Compute ROC curves ─────────────────────────────────────────────────────────
eps_grid = np.linspace(0.2, 1.0, 300)

for name, m in models.items():
    scores = m["scores_sel"]
    fpr, tpr, _ = roc_curve(labels_sel, scores, sample_weight=weights_sel)
    roc_auc = auc(fpr, tpr)
    if roc_auc < 0.5:   # flip orientation if needed
        scores = -scores
        fpr, tpr, _ = roc_curve(labels_sel, scores, sample_weight=weights_sel)
        roc_auc = auc(fpr, tpr)

    rej = np.where(fpr > 0, 1.0 / fpr, np.nan)
    rej_grid = np.interp(eps_grid, tpr, rej, left=np.nan, right=np.nan)
    valid = np.isfinite(rej_grid)

    m["fpr"] = fpr
    m["tpr"] = tpr
    m["rej"] = rej
    m["tpr_plot"] = eps_grid[valid]
    m["rej_plot"] = rej_grid[valid]
    m["auc"] = roc_auc

print("AUC scores:")
for name, m in models.items():
    print(f"  {name}: {m['auc']:.4f}")

In [ ]:
# ── Plot ROC with ratio panel ──────────────────────────────────────────────────
x_min, x_max = 0.2, 1.0

has_ref = REFERENCE_MODEL in models and "tpr_plot" in models[REFERENCE_MODEL]
fig, axes = plt.subplots(
    2 if has_ref else 1, 1,
    figsize=(7, 7 if has_ref else 5),
    dpi=150,
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05} if has_ref else {},
    sharex=True
)
ax1 = axes[0] if has_ref else axes
ax2 = axes[1] if has_ref else None

max_rej = 1.0
for i, (name, m) in enumerate(models.items()):
    color = MODEL_COLORS.get(name, default_colors[i % len(default_colors)])
    tpr_p, rej_p = m["tpr_plot"], m["rej_plot"]
    vis = (tpr_p >= x_min) & (tpr_p <= x_max) & np.isfinite(rej_p)
    if vis.any():
        max_rej = max(max_rej, np.nanmax(rej_p[vis]))
    ax1.plot(tpr_p, rej_p, label=f"{name}  (AUC = {m['auc']:.4f})", color=color, lw=2)

ax1.set_ylabel(r"Gluon rejection $\epsilon_g^{-1}$")
ax1.set_yscale("log")
ax1.set_ylim([1, max(10, 1.3 * max_rej)])
ax1.set_xlim([x_min, x_max])
ax1.legend(frameon=False)
ax1.xaxis.set_major_locator(MultipleLocator(0.1))
ax1.grid(True, which="major", ls=":", alpha=0.5)

# ATLAS-style label
ta = {"transform": ax1.transAxes, "va": "top"}
ax1.text(0.04, 0.97, "ATLAS",               fontsize=13, **ta, fontweight="bold", style="italic")
ax1.text(0.19, 0.97, "Simulation Internal",  fontsize=13, **ta)
ax1.text(0.04, 0.90, r"$\sqrt{s}=13$ TeV, SRJ", fontsize=11, **ta)
ax1.text(0.04, 0.83, f"{PT_MIN} < $p_T$ < {PT_MAX} GeV, {ETA_MIN} < |η| < {ETA_MAX}", fontsize=10, **ta)

if has_ref and ax2 is not None:
    ref = models[REFERENCE_MODEL]
    for i, (name, m) in enumerate(models.items()):
        color = MODEL_COLORS.get(name, default_colors[i % len(default_colors)])
        if name == REFERENCE_MODEL:
            ax2.axhline(1, color=color, ls="--", lw=1.5)
            continue
        rej_interp = np.interp(ref["tpr_plot"], m["tpr_plot"], m["rej_plot"],
                               left=np.nan, right=np.nan)
        ratio = rej_interp / ref["rej_plot"]
        valid = np.isfinite(ratio)
        ax2.plot(ref["tpr_plot"][valid], ratio[valid],
                 color=color, lw=1.8, label=f"{name} / {REFERENCE_MODEL}")
    ax2.set_ylabel(f"Ratio to\n{REFERENCE_MODEL}")
    ax2.set_xlabel(r"Quark efficiency $\epsilon_q$")
    ax2.set_ylim([0.5, 2.0])
    ax2.set_xlim([x_min, x_max])
    ax2.axhline(1, color="gray", ls=":", lw=1)
    ax2.xaxis.set_major_locator(MultipleLocator(0.1))
    ax2.grid(True, which="major", ls=":", alpha=0.5)
else:
    ax1.set_xlabel(r"Quark efficiency $\epsilon_q$")

plt.show()

## Rejection vs jet pT at fixed quark efficiency

In [ ]:
SIGNAL_EFFICIENCY = 0.5   # fix quark efficiency at 50%
PT_BINS = np.linspace(PT_MIN, PT_MAX, 8)

bin_centers     = (PT_BINS[:-1] + PT_BINS[1:]) / 2
bin_half_widths = (PT_BINS[1:]  - PT_BINS[:-1]) / 2

model_rejections = {}

for name, m in models.items():
    rejs = []
    for p_lo, p_hi in zip(PT_BINS[:-1], PT_BINS[1:]):
        mask = (pt_sel >= p_lo) & (pt_sel < p_hi)
        if mask.sum() < 10:
            rejs.append(np.nan)
            continue
        y_t = labels_sel[mask]
        y_s = m["scores_sel"][mask]
        w   = weights_sel[mask]
        if len(np.unique(y_t)) < 2:
            rejs.append(np.nan)
            continue
        # Threshold from signal at desired efficiency
        sig_s = y_s[y_t == 1]
        sig_w = w[y_t == 1]
        order = np.argsort(sig_s)
        cumw  = np.cumsum(sig_w[order])
        thr_idx = np.searchsorted(cumw, cumw[-1] * (1 - SIGNAL_EFFICIENCY))
        threshold = sig_s[order][min(thr_idx, len(sig_s) - 1)]
        # Background efficiency
        bkg_s = y_s[y_t == 0]
        bkg_w = w[y_t == 0]
        W_pass = bkg_w[bkg_s > threshold].sum()
        W_tot  = bkg_w.sum()
        rejs.append(1.0 / (W_pass / W_tot) if W_pass > 0 else np.nan)
    model_rejections[name] = np.array(rejs)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(7, 6), dpi=150,
    sharex=True, gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05}
)

for i, (name, rejs) in enumerate(model_rejections.items()):
    color = MODEL_COLORS.get(name, default_colors[i % len(default_colors)])
    valid = ~np.isnan(rejs)
    ax_top.errorbar(bin_centers[valid], rejs[valid], xerr=bin_half_widths[valid],
                    fmt="o", color=color, label=name, capsize=0, elinewidth=1.5)

ax_top.set_ylabel(r"Gluon rejection $\epsilon_g^{-1}$")
ax_top.legend(frameon=False)
ta = {"transform": ax_top.transAxes, "va": "top"}
ax_top.text(0.04, 0.97, "ATLAS",               fontsize=12, **ta, fontweight="bold", style="italic")
ax_top.text(0.19, 0.97, "Simulation Internal",  fontsize=12, **ta)
ax_top.text(0.04, 0.90, f"$\\epsilon_q = {int(SIGNAL_EFFICIENCY*100)}\\%$, {ETA_MIN} < |η| < {ETA_MAX}", fontsize=10, **ta)

if REFERENCE_MODEL in model_rejections:
    ref_rej = model_rejections[REFERENCE_MODEL]
    for i, (name, rejs) in enumerate(model_rejections.items()):
        color = MODEL_COLORS.get(name, default_colors[i % len(default_colors)])
        if name == REFERENCE_MODEL:
            ax_bot.axhline(1, color=color, ls="--", lw=1.5)
            continue
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = rejs / ref_rej
        valid = ~np.isnan(ratio)
        ax_bot.errorbar(bin_centers[valid], ratio[valid], xerr=bin_half_widths[valid],
                        fmt="o", color=color, capsize=0, elinewidth=1.5)

ax_bot.axhline(1, color="gray", ls=":", lw=1)
ax_bot.set_ylabel(f"Ratio to\n{REFERENCE_MODEL}")
ax_bot.set_xlabel(r"Jet $p_T$ [GeV]")
ax_bot.set_ylim([0.5, 2.0])
plt.show()

## Score correlations between models

In [ ]:
model_names = list(models.keys())
quark_mask = labels_sel == 1
gluon_mask = labels_sel == 0

pairs = [(a, b) for i, a in enumerate(model_names) for b in model_names[i+1:]]

for name_a, name_b in pairs:
    scores_a = models[name_a]["scores_sel"]
    scores_b = models[name_b]["scores_sel"]

    r_q = np.corrcoef(scores_a[quark_mask], scores_b[quark_mask])[0, 1]
    r_g = np.corrcoef(scores_a[gluon_mask], scores_b[gluon_mask])[0, 1]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=120)
    for ax, mask, title, cmap in [
        (ax1, gluon_mask, f"Gluons (r = {r_g:.3f})", "Blues"),
        (ax2, quark_mask, f"Quarks (r = {r_q:.3f})", "Reds"),
    ]:
        h = ax.hist2d(scores_a[mask], scores_b[mask],
                      bins=50, range=[[0,1],[0,1]], cmap=cmap, density=True)
        plt.colorbar(h[3], ax=ax, label="Density")
        ax.plot([0,1], [0,1], "k--", alpha=0.4)
        ax.set_xlabel(f"{name_a} score")
        ax.set_ylabel(f"{name_b} score")
        ax.set_title(title)
    plt.suptitle(f"{name_a} vs {name_b}")
    plt.tight_layout()
    plt.show()

## Confusion matrices

In [ ]:
THRESHOLD = 0.5   # classification threshold

for name, m in models.items():
    scores = m["scores_sel"]
    pred   = (scores > THRESHOLD).astype(int)

    cm = np.zeros((2, 2), dtype=int)
    for t in (0, 1):
        for p in (0, 1):
            cm[t, p] = int(((labels_sel == t) & (pred == p)).sum())

    fig, ax = plt.subplots(figsize=(5, 4), dpi=120)
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax)
    total = cm.sum()
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]:,}\n({100*cm[i,j]/total:.1f}%)",
                    ha="center", va="center", color="black")
    ax.set_xticks([0,1]); ax.set_xticklabels(["Pred gluon", "Pred quark"])
    ax.set_yticks([0,1]); ax.set_yticklabels(["Truth gluon", "Truth quark"])
    ax.set_xlabel(f"{name}  (threshold = {THRESHOLD})")
    ax.set_ylabel("Truth label")
    plt.tight_layout()
    plt.show()